# q2: GSM8K with Qwen2.5-0.5B — calculator agent
The model writes `CALC: <expression>`, a restricted calculator computes it (no `eval`/`exec`) and sends back `RESULT: <value>` or an error. At most 3 calls per question.

Run q1.ipynb first (it saves `q1_results.json`), then run all cells here in the same Colab runtime.

**Notes:**
- Same token limit, answer extraction and token counting as q1.
- Generation stops when the model starts writing its own `RESULT:` line, so it cannot make up results.
- The calculator also accepts small format differences (`3 x 4`, `20%`, `1,080`, `16 - 3 - 4 = 7`).


## 0. Install (only needed on Colab)

In [1]:
!pip install -q transformers datasets accelerate torch

## 1. Imports

In [2]:
import ast
import json
import operator as op
import re
from collections import Counter

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

## 2. Load model

In [3]:
name = "Qwen/Qwen2.5-0.5B-Instruct"

# pick the best available device automatically
if torch.cuda.is_available():
    device, dtype = "cuda", torch.float16
elif torch.backends.mps.is_available():
    device, dtype = "mps", torch.float16
else:
    device, dtype = "cpu", torch.float32
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype).to(device)

Using device: cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

## 3. Load data (dev first; section 10 switches to test)

In [4]:
ds = load_dataset("openai/gsm8k", "main")
train, test = ds["train"], ds["test"]

MODE = "dev"
questions = [train[i] for i in range(0, 10)]      # train 1-10
print(MODE, len(questions), "questions")


def gold(example):
    """The correct number is the text after '####' (only used for scoring, never shown to the model)."""
    return example["answer"].split("####")[-1].strip().replace(",", "")

dev 10 questions


## 4. Restricted calculator (no eval / exec)

In [5]:
OPS = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
       ast.FloorDiv: op.floordiv, ast.USub: op.neg, ast.UAdd: op.pos}


def evaluate(node):
    """Walk the parsed expression; anything except numbers and + - * / // ( ) is rejected."""
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        return OPS[type(node.op)](evaluate(node.left), evaluate(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in OPS:
        return OPS[type(node.op)](evaluate(node.operand))
    raise ValueError("not allowed")


def calculator(expr):
    """Returns a number, or an 'Error: ...' string the model can read."""
    expr = re.sub(r"(?<=\d),(?=\d{3}\b)", "", expr)                        # 1,080 -> 1080
    expr = re.sub(r"\\frac\{([^{}]*)\}\{([^{}]*)\}", r"(\1)/(\2)", expr)   # \frac{a}{b} -> (a)/(b)
    for a, b in [("$", ""), ("\\(", ""), ("\\)", ""), ("\\[", ""), ("\\]", ""), ("\\times", "*"),
                 ("\\cdot", "*"), ("\\div", "/"), ("×", "*"), ("÷", "/"), ("−", "-")]:
        expr = expr.replace(a, b)
    expr = re.sub(r"(?<=[\d)])\s*[xX]\s*(?=[\d(])", "*", expr)              # 3 x 4 -> 3*4
    expr = re.sub(r"(\d+(?:\.\d+)?)\s*%", r"(\1/100)", expr)                 # 20% -> (20/100)
    error = "empty expression"
    for part in expr.split("="):                                            # "16 - 3 - 4 = 7" -> "16 - 3 - 4"
        part = part.strip().rstrip(".")
        if not part:
            continue
        try:
            value = evaluate(ast.parse(part, mode="eval").body)
            if abs(value) > 1e15:
                raise ValueError("too large")
            return int(value) if float(value).is_integer() else round(value, 4)
        except ZeroDivisionError:
            error = "division by zero"
        except Exception:
            error = "invalid expression, use only numbers and + - * / ( )"
    return f"Error: {error}"


for e in ["48 + 48 / 2", "12 * 50 / 60", "16 - 3 - 4 = 7", "3 x 4", "20% * 50", "1 / 0", "2 ** 3", "__import__('os')"]:
    print(f"{e!r:<22} -> {calculator(e)}")

'48 + 48 / 2'          -> 72
'12 * 50 / 60'         -> 10
'16 - 3 - 4 = 7'       -> 9
'3 x 4'                -> 12
'20% * 50'             -> 10
'1 / 0'                -> Error: division by zero
'2 ** 3'               -> Error: invalid expression, use only numbers and + - * / ( )
"__import__('os')"     -> Error: invalid expression, use only numbers and + - * / ( )


## 5. Prompt (tuned on dev only)

In [6]:
SYSTEM = """Solve the math problem. Use the calculator for every computation.
To use it, write one line: CALC: <arithmetic expression>
Then stop. You will get back: RESULT: <value>
Use only numbers and + - * / ( ). You can use the calculator at most 3 times.
When you know the answer, write: The answer is <number>.

Example:
Question: Tom has 3 boxes with 12 apples each. He eats 5 apples. How many apples are left?
Apples in the boxes are 3 * 12, minus the 5 he ate.
CALC: 3 * 12 - 5
RESULT: 31
The answer is 31."""

LAST = "No calculator calls left. Write: The answer is <number>."

## 6. Answer parsing (same as q1)

In [7]:
NUM = r"-?\d+(?:\.\d+)?"


def canon(x):
    """'18.00' -> '18', so equal numbers are equal strings (matters for majority voting)."""
    v = float(x)
    return str(int(v)) if v.is_integer() else str(round(v, 4))


def get_number(text):
    """Final answer: 'The answer is X', else \\boxed{X}, else from the last line that has a number:
    the first number after its last '=' / 'is', else its first number ('earned $10 for 50 minutes' -> 10)."""
    text = re.sub(r"(?<=\d),(?=\d{3}\b)", "", text).replace("$", "")      # 1,080 -> 1080, $5 -> 5
    m = (re.findall(r"answer is[\s:*\\(]*(" + NUM + ")", text, re.I)
         or re.findall(r"\\boxed\{\D{0,10}?(" + NUM + ")", text))
    if m:
        return canon(m[-1])
    lines = [re.sub(r"^\s*\d+[.)]\s", "", l) for l in text.splitlines()]   # drop "3. " list numbers
    lines = [l for l in lines if re.search(r"\d", l)]
    if not lines:
        return None
    after_eq = lines[-1].split("=")[-1]
    after_is = re.split(r"\b(?:is|are|was|were)\b", after_eq)[-1]
    for part in (after_is, after_eq, lines[-1]):
        nums = re.findall(NUM, part)
        if nums:
            return canon(nums[0])


def same(a, b):
    try:
        return a is not None and abs(float(a) - float(b)) < 1e-6
    except (TypeError, ValueError):
        return False

## 7. Model call + the agent

In [8]:
MAX_NEW_TOKENS = 512   # same as q1

eos = model.generation_config.eos_token_id
END_IDS = set(eos if isinstance(eos, list) else [eos]) | {tokenizer.eos_token_id, tokenizer.pad_token_id}

tokens_used = 0        # total generated tokens
calls_made = 0         # total model calls (one per generated solution)
texts = []             # (output, n_tokens) of the current question, saved by the runner


def generate_texts(messages, **settings):
    """One generate() call. Every returned solution counts as one model call."""
    global tokens_used, calls_made
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True, return_dict=True
    ).to(device)
    n = inputs["input_ids"].shape[-1]
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, **settings)
    outputs = []
    for seq in out[:, n:].tolist():
        k = next((i + 1 for i, t in enumerate(seq) if t in END_IDS), len(seq))   # up to the end token
        text = tokenizer.decode(seq[:k], skip_special_tokens=True)
        tokens_used += k
        calls_made += 1
        texts.append((text, k))
        outputs.append(text)
    return outputs


def ask(messages):
    """One greedy model call. Stops if the model starts writing its own RESULT line."""
    return generate_texts(messages, do_sample=False, temperature=None, top_p=None, top_k=None,
                          stop_strings=["\nRESULT:"], tokenizer=tokenizer)[0]


def agent(q):
    """Think -> CALC -> observe RESULT, at most 3 calculator calls. Returns (answer, steps, transcript)."""
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    steps = []                                                    # (expression, result) pairs
    while True:
        reply = ask(msgs)
        m = re.search(r"CALC:([^\n]*)", reply, re.I)
        if not m or len(steps) == 3:                              # final answer (or out of calls)
            return get_number(reply), steps, msgs[2:] + [{"role": "assistant", "content": reply}]
        expr = m.group(1).split("RESULT")[0].strip()              # drop a RESULT written on the same line
        result = calculator(expr)
        steps.append((expr, result))
        msgs += [{"role": "assistant", "content": reply[:m.start()] + "CALC: " + expr},
                 {"role": "user", "content": f"RESULT: {result}" + (f"\n{LAST}" if len(steps) == 3 else "")}]

## 8. Runner + `show_failures`
`run_agent` saves every question's steps and full conversation. `show_failures` only prints what was saved — it does **not** run the model again.
- **formulation**: the answer equals the last calculator result → the arithmetic was right, the expression was wrong
- **arithmetic / copy**: the answer differs from the last calculator result → the model mis-copied or ignored it
- **calculator error**: the last calculator call failed and the model answered anyway
- **no calculator used**: the model did all the math itself

In [9]:
results = {}   # mode -> (correct, tokens, model calls, calculator calls)
records = {}   # mode -> per-question records


def run_agent():
    global tokens_used, calls_made
    tokens_used, calls_made = 0, 0
    records[MODE] = []
    for i, ex in enumerate(questions, 1):
        texts.clear()
        pred, steps, transcript = agent(ex["question"])
        ok = same(pred, gold(ex))
        records[MODE].append({"i": i, "ex": ex, "pred": pred, "ok": ok, "steps": steps, "transcript": transcript})
        print(f"{i}. pred={pred} gold={gold(ex)} {'OK' if ok else 'WRONG'}  steps={steps}")
    correct = sum(r["ok"] for r in records[MODE])
    tool_calls = sum(len(r["steps"]) for r in records[MODE])
    results[MODE] = (correct, tokens_used, calls_made, tool_calls)
    print(f"\nagent ({MODE}): {correct}/{len(questions)}  tokens={tokens_used}  "
          f"model calls={calls_made}  calculator calls={tool_calls}")


def failure_type(r):
    if not r["steps"]:
        return "no calculator used"
    last = r["steps"][-1][1]
    if isinstance(last, str):
        return "calculator error"
    return "formulation" if same(r["pred"], last) else "arithmetic / copy"


def show_failures():
    wrong = [r for r in records[MODE] if not r["ok"]]
    print(Counter(failure_type(r) for r in wrong), "\n")
    for r in wrong:
        print(f"========== Q{r['i']}  pred={r['pred']}  gold={gold(r['ex'])}  [{failure_type(r)}] ==========")
        print("QUESTION:", r["ex"]["question"])
        for m in r["transcript"]:
            print(f"--- {m['role']} ---\n{m['content']}")
        print()

## 9. Dev run (train 1-10)

In [10]:
run_agent()

1. pred=72 gold=72 OK  steps=[('48 / 2', 24)]
2. pred=10 gold=10 OK  steps=[('12 * 50 / 60', 10)]
3. pred=5 gold=5 OK  steps=[('(1/2) * 100 - 15 - 2 * 15', 5)]
4. pred=42 gold=42 OK  steps=[]
5. pred=312 gold=624 WRONG  steps=[]
6. pred=35 gold=35 OK  steps=[]
7. pred=48 gold=48 OK  steps=[('(2 * 16) + (2 * 8)', 48)]
8. pred=24 gold=16 WRONG  steps=[('2 * 3 * 2 * 2', 24)]
9. pred=41 gold=41 OK  steps=[]
10. pred=360 gold=990 WRONG  steps=[('10 * 18 + 10 * 18 * .5 + 10 * 18 * .5', 360)]

agent (dev): 7/10  tokens=888  model calls=16  calculator calls=6


In [11]:
show_failures()

Counter({'formulation': 2, 'no calculator used': 1}) 

========== Q5  pred=312  gold=624  [no calculator used] ==========
QUESTION: James writes a 3-page letter to 2 different friends twice a week.  How many pages does he write a year?
--- assistant ---
James writes a 3-page letter to 2 different friends twice a week, so he writes 3 x 2 = 6 letters per week.
There are 52 weeks in a year, so he writes 6 x 52 = 312 letters in a year.
Therefore, James writes 312 pages in a year.

The answer is 312.

========== Q8  pred=24  gold=16  [formulation] ==========
QUESTION: Ken created a care package to send to his brother, who was away at boarding school.  Ken placed a box on a scale, and then he poured into the box enough jelly beans to bring the weight to 2 pounds.  Then, he added enough brownies to cause the weight to triple.  Next, he added another 2 pounds of jelly beans.  And finally, he added enough gummy worms to double the weight once again.  What was the final weight of the box of good

## 10. Final test run (test 1-30, prompt fixed on dev)
Run the switch cell once. Do **not** re-run sections 3 or 8 after this, or everything goes back to dev.

In [12]:
if MODE == "dev":                                # safe to re-run: only switches once
    MODE = "test"
    questions = [test[i] for i in range(0, 30)]  # test 1-30
print(MODE, len(questions), "questions")

test 30 questions


In [13]:
run_agent()

1. pred=9 gold=18 WRONG  steps=[('16 - 3 - 4 = 7', 9)]
2. pred=5 gold=3 WRONG  steps=[]
3. pred=-170000 gold=70000 WRONG  steps=[]
4. pred=540 gold=540 OK  steps=[('3 * 3 * 60', 540)]
5. pred=120 gold=20 WRONG  steps=[('(15 + 25) * 3', 120)]
6. pred=128 gold=64 WRONG  steps=[('16 * 5 + 16 * 5 * 0.6', 128)]
7. pred=100 gold=260 WRONG  steps=[('4 * 20 + 20 = 100', 100)]
8. pred=0 gold=160 WRONG  steps=[('200 / 2 + 20', 120)]
9. pred=243.5 gold=45 WRONG  steps=[]
10. pred=940 gold=460 WRONG  steps=[('10 * 40 + 1.2 * 10 * 45', 940)]
11. pred=54 gold=366 WRONG  steps=[('60 * 3 - (60 * 3) * 0.7', 54.0)]
12. pred=8328 gold=694 WRONG  steps=[('(3 * 12) * 68 + (2 * 12) * 80 + (6 * 12) * 55', 8328)]
13. pred=11 gold=13 WRONG  steps=[]
14. pred=27 gold=18 WRONG  steps=[]
15. pred=60 gold=60 OK  steps=[]
16. pred=96 gold=125 WRONG  steps=[]
17. pred=4 gold=230 WRONG  steps=[]
18. pred=22750 gold=57500 WRONG  steps=[('(50 * 35) + (35 * 15)', 2275)]
19. pred=7 gold=7 OK  steps=[]
20. pred=8 gold=6 W

In [14]:
show_failures()

Counter({'formulation': 9, 'no calculator used': 7, 'arithmetic / copy': 4}) 

========== Q1  pred=9  gold=18  [formulation] ==========
QUESTION: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
--- assistant ---
CALC: 16 - 3 - 4 = 7
--- user ---
RESULT: 9
--- assistant ---
Janet makes $9 every day at the farmers' market.

========== Q2  pred=5  gold=3  [no calculator used] ==========
QUESTION: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
--- assistant ---
ROBE = 2 BOLTS OF BLUE FIBER + 0.5 * 2 BOLTS OF WHITE FIBER
ROBE = 4 BOLTS OF BLUE FIBER + 1 BOLT OF WHITE FIBER
ROBE = 5 BOLTS OF FIBER

Therefore, the answer is 5.

========== Q3  pred=-170000  gold=70000  [no calculator used] ==========
QUESTI

## 11. Comparison with the strongest Q1 method (test 1-30)

In [15]:
try:
    q1_test = json.load(open("q1_results.json"))["test"]                 # saved by section 12 of q1.ipynb
except FileNotFoundError:
    print("q1_results.json not found -> using the numbers typed here; check them against q1's test table\n")
    q1_test = {"four-shot": [10, 3446, 30]}
best = max(q1_test, key=lambda m: (q1_test[m][0], -q1_test[m][1]))     # most correct, then fewest tokens

c1, t1, k1 = q1_test[best]
c2, t2, k2, tools = results["test"]
print(f"{'method':<18}{'accuracy':<14}{'tokens':<10}{'model calls':<13}{'calculator calls':<16}")
print(f"{best:<18}{f'{c1}/30 ({100 * c1 / 30:.0f}%)':<14}{t1:<10}{k1:<13}{0:<16}")
print(f"{'calculator agent':<18}{f'{c2}/30 ({100 * c2 / 30:.0f}%)':<14}{t2:<10}{k2:<13}{tools:<16}")

method            accuracy      tokens    model calls  calculator calls
self-consistency  11/30 (37%)   15784     120          0               
calculator agent  10/30 (33%)   4028      50           20              
